# End-to-End Project: Recommendation System

Building a recommendation engine with multiple approaches.

## Project Overview

**Objective**: Build a movie recommendation system using collaborative and content-based filtering.

**Skills Applied**:
- Collaborative filtering (user-based and item-based)
- Content-based filtering
- Matrix factorization concepts
- Hybrid recommendation approaches
- Evaluation metrics for recommendations

In [ ]:
# Standard imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
from collections import defaultdict

# Sklearn imports
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

plt.style.use('seaborn-v0_8-whitegrid')
warnings.filterwarnings('ignore')
np.random.seed(42)

print("Libraries loaded successfully!")

## 1. Data Generation

We'll create synthetic movie rating and content data.

In [ ]:
def generate_movie_data(n_users=500, n_movies=100, n_ratings=10000):
    """
    Generate synthetic movie rating data with user preferences.
    """
    np.random.seed(42)
    
    # Movie metadata
    genres = ['Action', 'Comedy', 'Drama', 'Horror', 'Sci-Fi', 'Romance', 'Thriller']
    
    movies = pd.DataFrame({
        'movie_id': range(1, n_movies + 1),
        'title': [f"Movie {i}" for i in range(1, n_movies + 1)],
        'genre': np.random.choice(genres, n_movies),
        'year': np.random.randint(1990, 2024, n_movies),
        'popularity': np.random.uniform(0.3, 1.0, n_movies)
    })
    
    # Create user genre preferences (hidden)
    user_preferences = {}
    for user_id in range(1, n_users + 1):
        # Each user prefers 2-3 genres
        preferred_genres = np.random.choice(genres, np.random.randint(2, 4), replace=False)
        user_preferences[user_id] = set(preferred_genres)
    
    # Generate ratings based on preferences
    ratings_list = []
    
    for _ in range(n_ratings):
        user_id = np.random.randint(1, n_users + 1)
        movie_id = np.random.randint(1, n_movies + 1)
        
        # Get movie genre
        movie_genre = movies[movies['movie_id'] == movie_id]['genre'].values[0]
        
        # Base rating with preference boost
        if movie_genre in user_preferences[user_id]:
            base_rating = np.random.normal(4.0, 0.8)
        else:
            base_rating = np.random.normal(2.8, 1.0)
        
        # Add popularity influence
        popularity = movies[movies['movie_id'] == movie_id]['popularity'].values[0]
        rating = base_rating + (popularity - 0.5) * 0.5
        
        # Clip to valid range
        rating = np.clip(rating, 1, 5)
        
        ratings_list.append({
            'user_id': user_id,
            'movie_id': movie_id,
            'rating': round(rating, 1)
        })
    
    ratings = pd.DataFrame(ratings_list)
    
    # Remove duplicate user-movie pairs, keep last
    ratings = ratings.drop_duplicates(subset=['user_id', 'movie_id'], keep='last')
    
    return movies, ratings, user_preferences


# Generate data
movies, ratings, user_preferences = generate_movie_data()

print(f"Movies: {len(movies)}")
print(f"Ratings: {len(ratings)}")
print(f"Users: {ratings['user_id'].nunique()}")
print(f"\nRating statistics:")
print(ratings['rating'].describe())

In [ ]:
# Display sample data
print("=== Movies Sample ===")
print(movies.head())

print("\n=== Ratings Sample ===")
print(ratings.head())

In [ ]:
# Visualize data distribution
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Rating distribution
axes[0, 0].hist(ratings['rating'], bins=20, edgecolor='black')
axes[0, 0].set_xlabel('Rating')
axes[0, 0].set_ylabel('Count')
axes[0, 0].set_title('Rating Distribution')

# Ratings per user
user_counts = ratings.groupby('user_id').size()
axes[0, 1].hist(user_counts, bins=30, edgecolor='black')
axes[0, 1].set_xlabel('Number of Ratings')
axes[0, 1].set_ylabel('Number of Users')
axes[0, 1].set_title('Ratings per User')

# Genre distribution
genre_counts = movies['genre'].value_counts()
axes[1, 0].bar(genre_counts.index, genre_counts.values)
axes[1, 0].set_xlabel('Genre')
axes[1, 0].set_ylabel('Count')
axes[1, 0].set_title('Movies per Genre')
axes[1, 0].tick_params(axis='x', rotation=45)

# Average rating by genre
movie_ratings = ratings.merge(movies[['movie_id', 'genre']], on='movie_id')
genre_avg = movie_ratings.groupby('genre')['rating'].mean().sort_values()
axes[1, 1].barh(genre_avg.index, genre_avg.values)
axes[1, 1].set_xlabel('Average Rating')
axes[1, 1].set_title('Average Rating by Genre')

plt.tight_layout()
plt.show()

## 2. Create User-Item Matrix

In [ ]:
# Create user-item rating matrix
user_item_matrix = ratings.pivot_table(
    index='user_id',
    columns='movie_id',
    values='rating',
    fill_value=0
)

print(f"User-Item Matrix Shape: {user_item_matrix.shape}")
print(f"Sparsity: {(user_item_matrix == 0).sum().sum() / user_item_matrix.size * 100:.1f}%")

# Display sample
user_item_matrix.iloc[:5, :5]

In [ ]:
# Visualize a portion of the matrix
plt.figure(figsize=(12, 8))
sns.heatmap(user_item_matrix.iloc[:50, :30], cmap='YlOrRd', cbar_kws={'label': 'Rating'})
plt.xlabel('Movie ID')
plt.ylabel('User ID')
plt.title('User-Item Rating Matrix (Sample)')
plt.tight_layout()
plt.show()

## 3. Collaborative Filtering - User-Based

In [ ]:
class UserBasedCF:
    """
    User-based Collaborative Filtering recommender.
    
    Recommends items that similar users have liked.
    """
    
    def __init__(self, n_neighbors=20):
        self.n_neighbors = n_neighbors
        self.user_similarity = None
        self.user_item_matrix = None
        self.user_means = None
    
    def fit(self, user_item_matrix):
        """Compute user similarities."""
        self.user_item_matrix = user_item_matrix
        self.user_means = user_item_matrix.replace(0, np.nan).mean(axis=1)
        
        # Normalize by user mean for better similarity
        normalized = user_item_matrix.sub(self.user_means, axis=0).fillna(0)
        
        # Compute cosine similarity between users
        self.user_similarity = pd.DataFrame(
            cosine_similarity(normalized),
            index=user_item_matrix.index,
            columns=user_item_matrix.index
        )
        
        return self
    
    def predict(self, user_id, movie_id):
        """Predict rating for a user-movie pair."""
        if user_id not in self.user_item_matrix.index:
            return self.user_means.mean()
        
        if movie_id not in self.user_item_matrix.columns:
            return self.user_means[user_id]
        
        # Get similar users who rated this movie
        similarities = self.user_similarity[user_id]
        movie_ratings = self.user_item_matrix[movie_id]
        
        # Filter to users who rated this movie
        rated_mask = movie_ratings > 0
        rated_similarities = similarities[rated_mask]
        rated_ratings = movie_ratings[rated_mask]
        
        if len(rated_ratings) == 0:
            return self.user_means[user_id]
        
        # Get top N similar users
        top_n = rated_similarities.nlargest(self.n_neighbors)
        top_ratings = rated_ratings[top_n.index]
        
        # Weighted average
        if top_n.sum() == 0:
            return self.user_means[user_id]
        
        prediction = (top_n * top_ratings).sum() / top_n.sum()
        return np.clip(prediction, 1, 5)
    
    def recommend(self, user_id, n_recommendations=10):
        """Get top N recommendations for a user."""
        if user_id not in self.user_item_matrix.index:
            # Cold start: recommend popular items
            return self.user_item_matrix.mean().nlargest(n_recommendations).index.tolist()
        
        # Get items user hasn't rated
        user_ratings = self.user_item_matrix.loc[user_id]
        unrated_items = user_ratings[user_ratings == 0].index
        
        # Predict ratings for unrated items
        predictions = {}
        for movie_id in unrated_items:
            predictions[movie_id] = self.predict(user_id, movie_id)
        
        # Sort and return top N
        sorted_predictions = sorted(predictions.items(), key=lambda x: x[1], reverse=True)
        return [item[0] for item in sorted_predictions[:n_recommendations]]


# Train user-based CF
user_cf = UserBasedCF(n_neighbors=20)
user_cf.fit(user_item_matrix)

print("User-Based CF trained!")
print(f"User similarity matrix shape: {user_cf.user_similarity.shape}")

In [ ]:
# Test user-based recommendations
test_user = 1
recommendations = user_cf.recommend(test_user, n_recommendations=5)

print(f"=== Recommendations for User {test_user} (User-Based CF) ===")
print(f"User's preferred genres: {user_preferences[test_user]}")
print("\nRecommended movies:")
for movie_id in recommendations:
    movie = movies[movies['movie_id'] == movie_id].iloc[0]
    pred_rating = user_cf.predict(test_user, movie_id)
    print(f"  {movie['title']} ({movie['genre']}) - Predicted: {pred_rating:.2f}")

## 4. Collaborative Filtering - Item-Based

In [ ]:
class ItemBasedCF:
    """
    Item-based Collaborative Filtering recommender.
    
    Recommends items similar to what the user has liked.
    """
    
    def __init__(self, n_neighbors=20):
        self.n_neighbors = n_neighbors
        self.item_similarity = None
        self.user_item_matrix = None
    
    def fit(self, user_item_matrix):
        """Compute item similarities."""
        self.user_item_matrix = user_item_matrix
        
        # Transpose to get item-user matrix
        item_user_matrix = user_item_matrix.T
        
        # Compute cosine similarity between items
        self.item_similarity = pd.DataFrame(
            cosine_similarity(item_user_matrix),
            index=item_user_matrix.index,
            columns=item_user_matrix.index
        )
        
        return self
    
    def predict(self, user_id, movie_id):
        """Predict rating for a user-movie pair."""
        if user_id not in self.user_item_matrix.index:
            return 3.0
        
        if movie_id not in self.item_similarity.index:
            return 3.0
        
        # Get user's rated items
        user_ratings = self.user_item_matrix.loc[user_id]
        rated_items = user_ratings[user_ratings > 0]
        
        if len(rated_items) == 0:
            return 3.0
        
        # Get similarities to rated items
        similarities = self.item_similarity[movie_id][rated_items.index]
        
        # Get top N similar items
        top_n = similarities.nlargest(self.n_neighbors)
        top_ratings = rated_items[top_n.index]
        
        # Weighted average
        if top_n.sum() == 0:
            return rated_items.mean()
        
        prediction = (top_n * top_ratings).sum() / top_n.sum()
        return np.clip(prediction, 1, 5)
    
    def recommend(self, user_id, n_recommendations=10):
        """Get top N recommendations for a user."""
        if user_id not in self.user_item_matrix.index:
            return self.user_item_matrix.mean().nlargest(n_recommendations).index.tolist()
        
        # Get items user hasn't rated
        user_ratings = self.user_item_matrix.loc[user_id]
        unrated_items = user_ratings[user_ratings == 0].index
        
        # Predict ratings for unrated items
        predictions = {}
        for movie_id in unrated_items:
            predictions[movie_id] = self.predict(user_id, movie_id)
        
        # Sort and return top N
        sorted_predictions = sorted(predictions.items(), key=lambda x: x[1], reverse=True)
        return [item[0] for item in sorted_predictions[:n_recommendations]]
    
    def get_similar_items(self, movie_id, n=5):
        """Find similar items to a given item."""
        if movie_id not in self.item_similarity.index:
            return []
        
        similarities = self.item_similarity[movie_id].drop(movie_id)
        return similarities.nlargest(n).index.tolist()


# Train item-based CF
item_cf = ItemBasedCF(n_neighbors=20)
item_cf.fit(user_item_matrix)

print("Item-Based CF trained!")
print(f"Item similarity matrix shape: {item_cf.item_similarity.shape}")

In [ ]:
# Test item-based recommendations
recommendations = item_cf.recommend(test_user, n_recommendations=5)

print(f"=== Recommendations for User {test_user} (Item-Based CF) ===")
print(f"User's preferred genres: {user_preferences[test_user]}")
print("\nRecommended movies:")
for movie_id in recommendations:
    movie = movies[movies['movie_id'] == movie_id].iloc[0]
    pred_rating = item_cf.predict(test_user, movie_id)
    print(f"  {movie['title']} ({movie['genre']}) - Predicted: {pred_rating:.2f}")

In [ ]:
# Find similar items
sample_movie = 1
similar_movies = item_cf.get_similar_items(sample_movie, n=5)

print(f"=== Movies Similar to Movie {sample_movie} ===")
original = movies[movies['movie_id'] == sample_movie].iloc[0]
print(f"Original: {original['title']} ({original['genre']})")
print("\nSimilar movies:")
for movie_id in similar_movies:
    movie = movies[movies['movie_id'] == movie_id].iloc[0]
    similarity = item_cf.item_similarity.loc[sample_movie, movie_id]
    print(f"  {movie['title']} ({movie['genre']}) - Similarity: {similarity:.3f}")

## 5. Content-Based Filtering

In [ ]:
class ContentBasedRecommender:
    """
    Content-based recommender using item features.
    
    Recommends items with similar content to user's liked items.
    """
    
    def __init__(self):
        self.item_features = None
        self.item_similarity = None
        self.user_item_matrix = None
        self.movies = None
    
    def fit(self, movies, user_item_matrix):
        """Build item feature matrix and compute similarities."""
        self.movies = movies
        self.user_item_matrix = user_item_matrix
        
        # Create feature matrix from movie attributes
        # One-hot encode genres
        genre_dummies = pd.get_dummies(movies['genre'], prefix='genre')
        
        # Normalize year and popularity
        scaler = MinMaxScaler()
        normalized_features = pd.DataFrame(
            scaler.fit_transform(movies[['year', 'popularity']]),
            columns=['year_norm', 'popularity_norm']
        )
        
        # Combine features
        self.item_features = pd.concat([
            movies[['movie_id']].reset_index(drop=True),
            genre_dummies.reset_index(drop=True),
            normalized_features.reset_index(drop=True)
        ], axis=1).set_index('movie_id')
        
        # Compute content similarity
        self.item_similarity = pd.DataFrame(
            cosine_similarity(self.item_features),
            index=self.item_features.index,
            columns=self.item_features.index
        )
        
        return self
    
    def get_user_profile(self, user_id):
        """Build user profile from rated items."""
        if user_id not in self.user_item_matrix.index:
            return None
        
        user_ratings = self.user_item_matrix.loc[user_id]
        rated_items = user_ratings[user_ratings > 0]
        
        if len(rated_items) == 0:
            return None
        
        # Weighted average of item features by rating
        profile = np.zeros(len(self.item_features.columns))
        
        for movie_id, rating in rated_items.items():
            if movie_id in self.item_features.index:
                profile += rating * self.item_features.loc[movie_id].values
        
        profile /= rated_items.sum()
        
        return profile
    
    def predict(self, user_id, movie_id):
        """Predict rating based on content similarity to user profile."""
        user_profile = self.get_user_profile(user_id)
        
        if user_profile is None:
            return 3.0
        
        if movie_id not in self.item_features.index:
            return 3.0
        
        item_features = self.item_features.loc[movie_id].values
        
        # Cosine similarity between user profile and item
        similarity = cosine_similarity([user_profile], [item_features])[0, 0]
        
        # Scale to rating range
        return 1 + 4 * similarity
    
    def recommend(self, user_id, n_recommendations=10):
        """Get content-based recommendations."""
        if user_id not in self.user_item_matrix.index:
            return self.movies.nlargest(n_recommendations, 'popularity')['movie_id'].tolist()
        
        user_ratings = self.user_item_matrix.loc[user_id]
        unrated_items = user_ratings[user_ratings == 0].index
        
        predictions = {}
        for movie_id in unrated_items:
            predictions[movie_id] = self.predict(user_id, movie_id)
        
        sorted_predictions = sorted(predictions.items(), key=lambda x: x[1], reverse=True)
        return [item[0] for item in sorted_predictions[:n_recommendations]]


# Train content-based recommender
content_rec = ContentBasedRecommender()
content_rec.fit(movies, user_item_matrix)

print("Content-Based Recommender trained!")
print(f"Item features shape: {content_rec.item_features.shape}")
print(f"Features: {content_rec.item_features.columns.tolist()[:5]}...")

In [ ]:
# Test content-based recommendations
recommendations = content_rec.recommend(test_user, n_recommendations=5)

print(f"=== Recommendations for User {test_user} (Content-Based) ===")
print(f"User's preferred genres: {user_preferences[test_user]}")
print("\nRecommended movies:")
for movie_id in recommendations:
    movie = movies[movies['movie_id'] == movie_id].iloc[0]
    pred_rating = content_rec.predict(test_user, movie_id)
    print(f"  {movie['title']} ({movie['genre']}) - Predicted: {pred_rating:.2f}")

## 6. Hybrid Recommender

In [ ]:
class HybridRecommender:
    """
    Hybrid recommender combining multiple approaches.
    
    Weights different recommendation methods.
    """
    
    def __init__(self, user_cf, item_cf, content_rec, 
                 weights=(0.4, 0.4, 0.2)):
        self.user_cf = user_cf
        self.item_cf = item_cf
        self.content_rec = content_rec
        self.weights = weights
    
    def predict(self, user_id, movie_id):
        """Weighted prediction from all recommenders."""
        user_pred = self.user_cf.predict(user_id, movie_id)
        item_pred = self.item_cf.predict(user_id, movie_id)
        content_pred = self.content_rec.predict(user_id, movie_id)
        
        weighted = (
            self.weights[0] * user_pred +
            self.weights[1] * item_pred +
            self.weights[2] * content_pred
        )
        
        return np.clip(weighted, 1, 5)
    
    def recommend(self, user_id, n_recommendations=10):
        """Get hybrid recommendations."""
        user_item_matrix = self.user_cf.user_item_matrix
        
        if user_id not in user_item_matrix.index:
            return self.content_rec.recommend(user_id, n_recommendations)
        
        user_ratings = user_item_matrix.loc[user_id]
        unrated_items = user_ratings[user_ratings == 0].index
        
        predictions = {}
        for movie_id in unrated_items:
            predictions[movie_id] = self.predict(user_id, movie_id)
        
        sorted_predictions = sorted(predictions.items(), key=lambda x: x[1], reverse=True)
        return [item[0] for item in sorted_predictions[:n_recommendations]]


# Create hybrid recommender
hybrid_rec = HybridRecommender(
    user_cf, item_cf, content_rec,
    weights=(0.4, 0.4, 0.2)
)

# Test hybrid recommendations
recommendations = hybrid_rec.recommend(test_user, n_recommendations=5)

print(f"=== Recommendations for User {test_user} (Hybrid) ===")
print(f"User's preferred genres: {user_preferences[test_user]}")
print("\nRecommended movies:")
for movie_id in recommendations:
    movie = movies[movies['movie_id'] == movie_id].iloc[0]
    pred_rating = hybrid_rec.predict(test_user, movie_id)
    print(f"  {movie['title']} ({movie['genre']}) - Predicted: {pred_rating:.2f}")

## 7. Evaluation

In [ ]:
def evaluate_recommender(recommender, ratings_df, n_test=500):
    """
    Evaluate recommender using RMSE on held-out ratings.
    """
    # Sample test ratings
    test_sample = ratings_df.sample(n=min(n_test, len(ratings_df)), random_state=42)
    
    predictions = []
    actuals = []
    
    for _, row in test_sample.iterrows():
        pred = recommender.predict(row['user_id'], row['movie_id'])
        predictions.append(pred)
        actuals.append(row['rating'])
    
    predictions = np.array(predictions)
    actuals = np.array(actuals)
    
    rmse = np.sqrt(np.mean((predictions - actuals) ** 2))
    mae = np.mean(np.abs(predictions - actuals))
    
    return {'RMSE': rmse, 'MAE': mae}


# Evaluate all recommenders
print("=== Recommender Evaluation ===")

recommenders = {
    'User-Based CF': user_cf,
    'Item-Based CF': item_cf,
    'Content-Based': content_rec,
    'Hybrid': hybrid_rec
}

results = {}
for name, rec in recommenders.items():
    metrics = evaluate_recommender(rec, ratings)
    results[name] = metrics
    print(f"{name:20} RMSE: {metrics['RMSE']:.4f}  MAE: {metrics['MAE']:.4f}")

In [ ]:
# Visualize evaluation results
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(results))
width = 0.35

rmse_values = [results[r]['RMSE'] for r in results]
mae_values = [results[r]['MAE'] for r in results]

bars1 = ax.bar(x - width/2, rmse_values, width, label='RMSE')
bars2 = ax.bar(x + width/2, mae_values, width, label='MAE')

ax.set_ylabel('Error')
ax.set_title('Recommender Performance Comparison')
ax.set_xticks(x)
ax.set_xticklabels(list(results.keys()), rotation=15)
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Coverage and diversity analysis
def analyze_recommendations(recommender, user_ids, movies_df, n_recs=10):
    """
    Analyze coverage and diversity of recommendations.
    """
    all_recommendations = []
    genre_counts = defaultdict(int)
    
    for user_id in user_ids:
        recs = recommender.recommend(user_id, n_recommendations=n_recs)
        all_recommendations.extend(recs)
        
        for movie_id in recs:
            genre = movies_df[movies_df['movie_id'] == movie_id]['genre'].values[0]
            genre_counts[genre] += 1
    
    unique_items = len(set(all_recommendations))
    total_items = len(movies_df)
    coverage = unique_items / total_items
    
    return {
        'coverage': coverage,
        'unique_items': unique_items,
        'genre_distribution': dict(genre_counts)
    }


# Analyze hybrid recommender
sample_users = list(range(1, 51))  # First 50 users
analysis = analyze_recommendations(hybrid_rec, sample_users, movies)

print("=== Recommendation Analysis (Hybrid) ===")
print(f"Coverage: {analysis['coverage']:.2%}")
print(f"Unique items recommended: {analysis['unique_items']} / {len(movies)}")
print("\nGenre distribution in recommendations:")
for genre, count in sorted(analysis['genre_distribution'].items(), key=lambda x: x[1], reverse=True):
    print(f"  {genre}: {count}")

## 8. Production Interface

In [ ]:
class MovieRecommendationSystem:
    """
    Production-ready movie recommendation system.
    """
    
    def __init__(self, hybrid_recommender, movies_df):
        self.recommender = hybrid_recommender
        self.movies = movies_df
    
    def get_recommendations(self, user_id, n=10, include_details=True):
        """
        Get personalized recommendations for a user.
        """
        movie_ids = self.recommender.recommend(user_id, n_recommendations=n)
        
        if not include_details:
            return movie_ids
        
        recommendations = []
        for movie_id in movie_ids:
            movie = self.movies[self.movies['movie_id'] == movie_id].iloc[0]
            pred_rating = self.recommender.predict(user_id, movie_id)
            
            recommendations.append({
                'movie_id': int(movie_id),
                'title': movie['title'],
                'genre': movie['genre'],
                'year': int(movie['year']),
                'predicted_rating': round(pred_rating, 2)
            })
        
        return recommendations
    
    def get_similar_movies(self, movie_id, n=5):
        """
        Get movies similar to a given movie.
        """
        similar_ids = self.recommender.item_cf.get_similar_items(movie_id, n=n)
        
        similar_movies = []
        for mid in similar_ids:
            movie = self.movies[self.movies['movie_id'] == mid].iloc[0]
            similarity = self.recommender.item_cf.item_similarity.loc[movie_id, mid]
            
            similar_movies.append({
                'movie_id': int(mid),
                'title': movie['title'],
                'genre': movie['genre'],
                'similarity': round(similarity, 3)
            })
        
        return similar_movies
    
    def display_recommendations(self, user_id, n=5):
        """
        Display formatted recommendations.
        """
        recs = self.get_recommendations(user_id, n=n)
        
        print(f"\n{'='*60}")
        print(f"  🎬 Recommendations for User {user_id}")
        print(f"{'='*60}")
        
        for i, rec in enumerate(recs, 1):
            stars = '⭐' * int(round(rec['predicted_rating']))
            print(f"\n  {i}. {rec['title']}")
            print(f"     Genre: {rec['genre']} | Year: {rec['year']}")
            print(f"     Predicted Rating: {rec['predicted_rating']} {stars}")
        
        print(f"\n{'='*60}")


# Create production system
rec_system = MovieRecommendationSystem(hybrid_rec, movies)

# Demo
rec_system.display_recommendations(test_user, n=5)

In [ ]:
# Get JSON-format recommendations (for API)
import json

api_response = rec_system.get_recommendations(test_user, n=3)
print("API Response:")
print(json.dumps(api_response, indent=2))

## 9. Save Models

In [ ]:
import joblib

# Save recommendation system
model_path = Path('./models')
model_path.mkdir(exist_ok=True)

model_data = {
    'user_cf': {
        'user_similarity': user_cf.user_similarity,
        'user_item_matrix': user_cf.user_item_matrix,
        'user_means': user_cf.user_means,
        'n_neighbors': user_cf.n_neighbors
    },
    'item_cf': {
        'item_similarity': item_cf.item_similarity,
        'user_item_matrix': item_cf.user_item_matrix,
        'n_neighbors': item_cf.n_neighbors
    },
    'content_rec': {
        'item_features': content_rec.item_features,
        'item_similarity': content_rec.item_similarity,
        'user_item_matrix': content_rec.user_item_matrix
    },
    'hybrid_weights': hybrid_rec.weights,
    'movies': movies,
    'evaluation': results
}

joblib.dump(model_data, model_path / 'recommendation_system.joblib')
print(f"Recommendation system saved to {model_path / 'recommendation_system.joblib'}")

## Summary

### Skills Demonstrated

- **Collaborative Filtering**: User-based and item-based approaches
- **Content-Based Filtering**: Feature engineering from item metadata
- **Hybrid Systems**: Combining multiple recommendation methods
- **Similarity Metrics**: Cosine similarity for users and items
- **Evaluation**: RMSE, MAE, coverage analysis
- **Production Interface**: Clean API for serving recommendations

### Key Takeaways

1. User-based CF finds users with similar preferences
2. Item-based CF finds similar items to user's history
3. Content-based uses item features for recommendations
4. Hybrid systems combine strengths of multiple approaches
5. Evaluation should include accuracy AND diversity metrics